# 05 — Normalize, Export + Inspect Concepts Over Time (Stages 10–12 + rollup)
**Does:** (a) L2-normalized vector exports (`.kv`, originals untouched); (b) descriptive diachronic concept viewer; (c) final `RUN_SUMMARY.md` rollup of the whole pipeline. No regressions, no p-values — trajectories with stability bands only.
**Reads:** trained `.model` files + `training_manifest.csv`. Models are loaded one at a time and released.


In [ ]:
# Cell 1 — RUN INPUT (the only cells you must edit: this one for vectors, Cell 4 for concepts).
DO_NORMALIZE = True
MODEL_FILTER_SUB = None
PERIOD_FILTER = None
CONCEPTS = [{"name": "EXAMPLE_replace_me", "targets": ["moderator"],
             "pole_a": ["fair", "helpful", "transparent"], "pole_b": ["biased", "corrupt", "abusive"],
             "anchors": ["community", "rules"]}]
K_NEIGHBORS = 20
SEED_FILTER = None
print(f"normalize={DO_NORMALIZE} concepts={len(CONCEPTS)} K={K_NEIGHBORS}")


In [ ]:
# Cell 2 — Setup: root, config, logger, dirs.
import os, sys, csv, json, gc, hashlib, datetime
from pathlib import Path
from collections import defaultdict
import yaml
import numpy as np
from src.paths import get_project_root
from src.storage import atomic_write_text, sha256_file, save_gensim_atomic

ROOT = get_project_root()
cfg = yaml.safe_load(open(ROOT / "config/project_config.yaml", encoding="utf-8"))
CLAIM_FLOOR = cfg["embeddings"].get("interpretation_floor", 100)
print("config", cfg["config_version"], "claim floor:", CLAIM_FLOOR)

try:
    from gensim.models import Word2Vec, KeyedVectors
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "gensim"])
    from gensim.models import Word2Vec, KeyedVectors

import logging
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOGP = ROOT / f"logs/05_vectors_inspect__{ts}__cfg-{cfg['config_version']}.log"
LOGP.parent.mkdir(parents=True, exist_ok=True)
lg = logging.getLogger("v5"); lg.setLevel(logging.INFO); lg.handlers.clear()
fh = logging.FileHandler(LOGP); fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
sh = logging.StreamHandler(sys.stdout); sh.setLevel(logging.WARNING)
lg.addHandler(fh); lg.addHandler(sh)

for d in ["vectors", "diagnostics/semantic_axes"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

today = datetime.datetime.now(datetime.timezone.utc).date().isoformat()
def atomic_text(path: Path, text: str): atomic_write_text(path, text)
def sha_of(p: Path): return sha256_file(p)
OUTD = ROOT / "diagnostics/semantic_axes"
print("setup ready")

In [ ]:
# Cell 3 — NORMALIZE + EXPORT: KeyedVectors with fill_norms; originals never overwritten.
# Safe atomic save handles companion .npy files without breaking reload.
TMAN = ROOT / "manifests/training_manifest.csv"
rows = list(csv.DictReader(open(TMAN, encoding="utf-8"))) if TMAN.exists() else []
T_COLS = list(rows[0].keys()) if rows else []
n_norm = n_skip = n_fail = 0

if DO_NORMALIZE:
    for r in rows:
        if r["status"] != "complete": continue
        if MODEL_FILTER_SUB and r["subreddit_or_group"] != MODEL_FILTER_SUB: continue
        if PERIOD_FILTER and r["model_id"] not in PERIOD_FILTER: continue
        mp = Path(r["model_path"])
        vname = mp.stem.replace("w2v__", "vectors_norm__") + ".kv"
        vp = ROOT / "vectors" / vname
        if r.get("vectors_path") and Path(r["vectors_path"]).exists() and r.get("vectors_sha256"):
            if sha_of(Path(r["vectors_path"])) == r["vectors_sha256"]: n_skip += 1; continue
        try:
            m = Word2Vec.load(str(mp)); m.wv.fill_norms()
            save_gensim_atomic(m.wv, vp)
            assert vp.stat().st_size > 0
            r["vectors_path"], r["vectors_sha256"] = str(vp), sha_of(vp)
            del m; gc.collect(); n_norm += 1
            if n_norm % 5 == 0: print(f"  normalized {n_norm}...")
        except Exception as e:
            lg.error(f"normalize {r['model_id']}: {e}"); n_fail += 1

    if T_COLS:
        tmp = TMAN.with_suffix(".tmp")
        f = open(tmp, "w", newline="", encoding="utf-8"); w = csv.DictWriter(f, fieldnames=T_COLS)
        w.writeheader(); w.writerows(rows); f.flush(); os.fsync(f.fileno()); f.close(); os.replace(tmp, TMAN)

print(f"normalize: new={n_norm} skipped={n_skip} failed={n_fail}")
print("safe export complete; KeyedVectors ready in vectors/")

In [ ]:
# Cell 4 — CONCEPT INPUT is Cell 1 (CONCEPTS). Here: index models grouped by (group, period).
# Prefers loading lightweight KeyedVectors (.kv) when available.
groups = defaultdict(list)
for r in rows:
    if r["status"] != "complete": continue
    if MODEL_FILTER_SUB and r["subreddit_or_group"] != MODEL_FILTER_SUB: continue
    if SEED_FILTER and int(r["seed"]) not in SEED_FILTER: continue
    
    vp = Path(r.get("vectors_path", "")) if r.get("vectors_path") else None
    mp = Path(r["model_path"])
    target_p = vp if (vp and vp.exists() and vp.stat().st_size > 0) else mp
    if target_p.exists() and target_p.stat().st_size > 0:
        groups[(r["subreddit_or_group"], r["period_id"])].append((int(r["seed"]), target_p, target_p == vp))

for k in groups: groups[k].sort()
order = sorted(groups)
print(f"cells: {len(groups)} models: {sum(len(v) for v in groups.values())}")
if not groups: print("STOP: no complete models — run Notebook 04 first.")

In [ ]:
# Cell 5 — METRIC ENGINE (per model, normalized first): coverage, coherence, separation, axis,
# projections, neighbor sets, Jaccard (within-period across seeds; consecutive-period same seed), frequencies.
def cos(a, b):
    d = float(np.linalg.norm(a) * np.linalg.norm(b))
    return float(np.dot(a, b) / d) if d else 0.0

def mean_pairwise(vecs):
    if len(vecs) < 2: return None
    s = n = 0.0
    for i in range(len(vecs)):
        for j in range(i + 1, len(vecs)):
            s += cos(vecs[i], vecs[j]); n += 1
    return s / n

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 0.0

def analyze_model(wv, concept, K):
    wv.fill_norms()
    out = {"coverage": {}, "coherence": {}, "freq": {}}; present = {}
    for pole in ("pole_a", "pole_b"):
        ok, miss = [], []
        for w in concept.get(pole, []) or []:
            if w in wv:
                c = int(wv.get_vecattr(w, "count")) if hasattr(wv, "get_vecattr") else 1000
                out["freq"][w] = c
                (ok if c >= CLAIM_FLOOR else miss).append(w)
            else: miss.append(w)
        present[pole] = ok; out["coverage"][pole] = {"kept": ok, "missing_or_rare": miss}
    for w in (concept.get("targets", []) + concept.get("anchors", [])):
        out["freq"][w] = int(wv.get_vecattr(w, "count")) if (hasattr(wv, "get_vecattr") and w in wv) else (1000 if w in wv else 0)
    va = [wv.get_vector(w, norm=True) for w in present["pole_a"]]
    vb = [wv.get_vector(w, norm=True) for w in present["pole_b"]]
    out["coherence"]["pole_a"] = mean_pairwise(va); out["coherence"]["pole_b"] = mean_pairwise(vb)
    out["separation"] = float(np.mean([cos(a, b) for a in va for b in vb])) if va and vb else None
    axis = None
    if va and vb:
        axis = np.mean(va, axis=0) - np.mean(vb, axis=0)
        norm = np.linalg.norm(axis)
        axis = axis / norm if norm else None
    out["proj"] = {}
    for w in concept.get("targets", []):
        if w in wv and axis is not None:
            out["proj"][w] = float(np.dot(wv.get_vector(w, norm=True), axis))
        else: out["proj"][w] = None
    out["anchors"] = {}
    for a in concept.get("anchors", []):
        if a in wv:
            out["anchors"][a] = {w: cos(wv.get_vector(a, norm=True), wv.get_vector(w, norm=True)) for w in concept.get("targets", []) if w in wv}
    out["neighbors"] = {}
    for w in concept.get("targets", []):
        if w in wv:
            out["neighbors"][w] = [x for x, _ in wv.most_similar(w, topn=K)]
        else: out["neighbors"][w] = []
    return out

trows_out, cov_summary = [], {}
for c in CONCEPTS:
    cn = c["name"]
    for (grp, pid), seed_list in groups.items():
        by_seed = {}
        for (s, path, is_kv) in seed_list:
            try:
                wv = KeyedVectors.load(str(path)) if is_kv else Word2Vec.load(str(path)).wv
            except Exception as le:
                lg.warning(f"Failed to load {path}: {le}"); continue
            an = analyze_model(wv, c, K_NEIGHBORS)
            by_seed[s] = an; cov_summary[(cn, grp, pid, s)] = an["coverage"]
            for w, sc in an["proj"].items():
                trows_out.append([cn, grp, pid, s, "projection", w, sc, ""])
            for a, dt in an["anchors"].items():
                for w, sc in dt.items():
                    trows_out.append([cn, grp, pid, s, f"cos_anchor_{a}", w, sc, ""])
            for w, nb in an["neighbors"].items():
                trows_out.append([cn, grp, pid, s, "neighbors", w, "", ";".join(nb)])
            for w, cnt in an["freq"].items():
                trows_out.append([cn, grp, pid, s, "frequency", w, cnt, ""])
            for p, sc in an["coherence"].items():
                trows_out.append([cn, grp, pid, s, f"coherence_{p}", "", sc, ""])
            if an["separation"] is not None:
                trows_out.append([cn, grp, pid, s, "separation", "", an["separation"], ""])
            del wv; gc.collect()
        # Cross-seed Jaccard within period
        seeds_present = sorted(by_seed.keys())
        for w in c.get("targets", []):
            pair_j = []
            for i in range(len(seeds_present)):
                for j in range(i + 1, len(seeds_present)):
                    s1, s2 = seeds_present[i], seeds_present[j]
                    nb1, nb2 = by_seed[s1]["neighbors"].get(w, []), by_seed[s2]["neighbors"].get(w, [])
                    if nb1 and nb2: pair_j.append(jaccard(nb1, nb2))
            mean_j = float(np.mean(pair_j)) if pair_j else None
            trows_out.append([cn, grp, pid, "all_seeds", "cross_seed_neighbor_jaccard", w, mean_j, ""])

TRAJ = OUTD / "trajectories.csv"
tmp = TRAJ.with_suffix(".tmp")
f = open(tmp, "w", newline="", encoding="utf-8"); w = csv.writer(f)
w.writerow(["concept","subreddit_or_group","period_id","seed","metric","word_or_pole","value","extra"])
w.writerows(trows_out); f.flush(); os.fsync(f.fileno()); f.close(); os.replace(tmp, TRAJ)
print(f"trajectories.csv: {len(trows_out)} rows -> {TRAJ}")

In [ ]:
# Cell 6 — PLOTS: (A) neighbor Jaccard vs prev period per seed; (B) projection over time; (C) frequency guardrail.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
def series(metric, cn, grp, word):
    d = defaultdict(dict)
    for r in trows_out:
        if r[0] == cn and r[1] == grp and r[4] == metric and r[5] == word:
            try: d[r[2]][r[3]] = float(r[6])
            except ValueError: pass
    return d
def neighbors_of(cn, grp, pid, seed, t):
    for r in trows_out:
        if (r[0], r[1], r[2], r[3], r[4], r[5]) == (cn, grp, pid, seed, "neighbors", t):
            return r[7].split(";") if r[7] else []
    return []
made = []
for c in CONCEPTS:
    cn = c["name"]
    for grp in sorted({g for (g, p) in order}):
        pids = sorted({p for (g, p) in order if g == grp})
        if not pids: continue
        seeds = sorted({s for (g, p) in order if g == grp for (s, _) in groups[(g, p)]})
        for t in c.get("targets", []):
            fig, ax = plt.subplots(3, 1, figsize=(9, 10), sharex=True)
            for s in seeds:
                js, prev = [], None
                for pid in pids:
                    nb = neighbors_of(cn, grp, pid, s, t)
                    js.append((len(set(nb) & set(prev)) / len(set(nb) | set(prev)) if prev is not None and nb and prev else None))
                    prev = nb
                ax[0].plot(pids, js, marker="o", label=f"seed {s}")
            ax[0].set_title(f"{cn} / {t} @{grp}: neighbor overlap vs prev (Jaccard K={K_NEIGHBORS})")
            ax[0].set_ylim(-0.05, 1.05); ax[0].legend(fontsize=8); ax[0].grid(True, alpha=0.3)
            pj = series("projection", cn, grp, t)
            for s in seeds:
                ax[1].plot(pids, [pj.get(pid, {}).get(s) for pid in pids], marker="o", label=f"seed {s}")
            ax[1].set_title("axis projection over time (+ = pole_a side)"); ax[1].legend(fontsize=8); ax[1].grid(True, alpha=0.3)
            fq = series("frequency", cn, grp, t)
            for s in seeds[:1]:
                ax[2].plot(pids, [fq.get(pid, {}).get(s) for pid in pids], marker="o", color="black")
            ax[2].axhline(CLAIM_FLOOR, color="red", linestyle="--", label=f"claim floor ({CLAIM_FLOOR})")
            ax[2].set_title("target frequency (confound guardrail)"); ax[2].set_yscale("log")
            ax[2].legend(fontsize=8); ax[2].grid(True, alpha=0.3)
            plt.xticks(rotation=30); fig.tight_layout()
            pp = OUTD / f"traj__{cn}__{str(grp).lower()}__{t}.png"
            fig.savefig(pp, dpi=110); plt.close(fig); made.append(str(pp))
print(f"plots: {len(made)}")


In [ ]:
# Cell 7 — REPORTS + RUN_SUMMARY.md (whole-pipeline rollup: counts, registry, vectors, trajectories).
for c in CONCEPTS:
    cn = c["name"]
    L = [f"# Concept report: {cn} ({today}, cfg-{cfg['config_version']})", "",
         "Descriptive only — trajectories with stability bands. No regressions, no p-values.", ""]
    for grp in sorted({g for (g, p) in order}):
        pids = sorted({p for (g, p) in order if g == grp})
        L.append(f"## {grp} ({len(pids)} periods)")
        for pid in pids:
            cov = cov_summary.get((cn, grp, pid), {})
            if len(cov) < 2: L.append(f"- {pid}: WARNING single-seed cell — no stability band; provisional.")
            for pole in ("pole_a", "pole_b"):
                miss = set()
                for s, cv in cov.items(): miss.update(cv.get(pole, {}).get("missing_or_rare", []))
                if miss: L.append(f"- {pid}: {pole} missing/rare: {', '.join(sorted(miss))} (excluded from this period's axis).")
        for t in c.get("targets", []):
            fq = series("frequency", cn, grp, t)
            lo = [p for p in pids for s, v in fq.get(p, {}).items() if v < CLAIM_FLOOR]
            if lo: L.append(f"- {t}: below claim floor in {', '.join(lo)} — readings there are fragile.")
        L.append("")
    L.append("## How to claim a shift credibly")
    L.append("1. Same direction in >=2 seeds. 2. Coverage holds every period. 3. Frequency cannot explain it (Panel C flat while A/B move).")
    atomic_text(OUTD / f"report__{cn}.md", "\n".join(L) + "\n")
    print(f"report: report__{cn}.md")
n_models = sum(1 for r in rows if r["status"] == "complete")
n_vec = sum(1 for r in rows if r["status"] == "complete" and r.get("vectors_path"))
summary = [f"# RUN SUMMARY ({today}, config {cfg['config_version']})", "",
 f"- models complete: {n_models}; with normalized vectors: {n_vec}",
 f"- trajectory rows: {len(trows_out)}; plots: {len(made)}; concepts: {len(CONCEPTS)}",
 f"- corpus master: metadata/corpus_master.csv (refresh in Notebook 04)",
 f"- registries: manifests/retrieval_manifest.csv, manifests/shard_manifest.csv, manifests/training_manifest.csv",
 f"- rerun: any notebook; completes skip by checksum; failures retry-only", ""]
atomic_text(ROOT / "RUN_SUMMARY.md", "\n".join(summary))
print("\n".join(summary))
print("=" * 70)
print(f"DONE vectors={n_norm} concepts={len(CONCEPTS)} traj_rows={len(trows_out)} plots={len(made)}")
print("=" * 70)
